In [ ]:
from typing import *
from pathlib import Path
import gc

import warnings
import numpy as np
import scipy
import torch
from torch import Tensor, nn
import pandas as pd
import json
from tqdm import tqdm

import matplotlib.pyplot as plt
from common.ml import ASVPreembeddedDataset
from gem.ml import safe_kl_div_loss, SGBAbundanceLayeredPredictionModel, SGBEmbedPoolConcatPredictionModel, SGBAbundancePredictionModel

# Baseline methods

In [1]:
class BaselinePredictor:
    def predict_abundances(self, asv_ids: List[str]) -> np.ndarray:
        raise NotImplementedError()

    def predict_batch_abundances(self, batched_inputs: List[List[str]]) -> np.ndarray:
        max_asvs = max(len(asv_list) for asv_list in batched_inputs)
        return np.stack([
            self.add_padding_1d(self.predict_abundances(batch), max_asvs)
            for batch in batched_inputs
        ], axis=0)

    @staticmethod
    def add_padding_1d(pred_vector: np.ndarray, max_size: int) -> np.ndarray:
        if max_size > len(pred_vector):
            pred_vector = np.concatenate([pred_vector, np.zeros(max_size - len(pred_vector), dtype=pred_vector.dtype)])
        return pred_vector

NameError: name 'List' is not defined

## Uniform abundance predictor

In [ ]:
""" Class definition. """
class UniformAbundancePredictor(BaselinePredictor):
    def __init__(self):
        pass

    def predict_abundances(self, asv_ids: List[str]) -> np.ndarray:
        """ Compute output. """
        pred = np.ones(len(asv_ids), dtype=float)
        pred = pred / np.sum(pred)
        return pred

## Nearest-Neighbor regression

In [ ]:
""" Class definition. """
class NearestNeighborAveragingPredictor(BaselinePredictor):
    """
    Predict an abundance by assigning a weight to each SGB in the specified query sample.
    The weight of SGB "X" is equal to the geometric mean of the abundance of NearestNeighbor(X) in the training samples.

    This class caches the result of input queries.

    Note: If "X" itself is in the training sample, then NearestNeighbor(X) = X, so we are counting the abundnaces of "X" itself in the training set.
    """
    def __init__(self, train_dset: ASVPreembeddedDataset):
        raise NotImplementedError("todo")

    def predict_abundances(self, asv_ids: List[str]) -> np.ndarray:
        raise NotImplementedError("todo")

# Model evaluation code

In [ ]:
# additional metrics
def bray_curtis(x: np.ndarray, y: np.ndarray):
    """
    Assume x, y are vectors which are identically indexed according to species (1, ..., n).
    Computes the Bray-curtis distance:
    BC = 1 - 2 * SUM_i[ min(x_i, y_i) ] / SUM_i [ x_i + y_i ]
    """
    numer = np.minimum(x, y).sum()
    denom = x.sum() + y.sum()
    return 1 - 2 * (numer / denom)


def evaluate_method(
    inference_fn: Callable[[ASVPreembeddedDataset, int], Tensor],
    dset: ASVPreembeddedDataset
) -> pd.DataFrame:
    """
    Function which evaluates the function (precisely how the evaluation is done, including pre & post-processing, is given by inference_fn) and outputs a DataFrame tabulating the results.

    :param inference_fn: A function (e.g. lambda expression) which takes a MicrobiomeSample as input and outputs a (abund_predictions) tensor. The abund_predictions should be a vector of LOGS of relative abundances.
    :param dset: The dataset to test on.
    """
    df_entries = []

    n_samples = len(dset)
    for sample_idx in tqdm(range(n_samples)):
        """ Prediction. """
        try:
            pred_asv_log_abunds: Tensor = inference_fn(dset, sample_idx)
        except Exception as e:
            print(f"During inference of sample {sample_idx}, got an error.")
            print(f"The error was: {e}")
            print(f"Skipping sample {sample_idx}")
            continue
        assert len(pred_asv_log_abunds.shape) == 1, "Expected a 1-d tensor for the true abundances."
        assert torch.logsumexp(pred_asv_log_abunds, dim=0).isclose(torch.tensor(0.0), rtol=1e-3, atol=1e-4).item(), f"Expected logsumexp(pred) to be equal to 0.0, got {torch.logsumexp(pred_asv_log_abunds, dim=0).item()}."
        # assert num_sgbs == pred_sgb_log_abunds.shape[0], "Pred abundances should have the same size/shape as the list of true SGB ids."

        """ Ground-truth. """
        _, true_asv_abunds = dset.true_abundance_profile(sample_idx)
        assert len(true_asv_abunds.shape) == 1, "Expected a 1-d tensor for the true abundances."
        assert true_asv_abunds.sum().isclose(torch.tensor(1.0), rtol=1e-4, atol=1e-7).item(), f"Expected true relative abundances to sum to 1.0, got {true_asv_abunds.sum()}"
        # assert num_sgbs == true_sgb_abunds.shape[0], "True abundances should have the same size/shape as the list of true SGB ids."

        # # DEBUG
        # print("Sample idx: ", sample_idx)
        # for pred, truth in zip(torch.exp(pred_sgb_log_abunds), true_sgb_abunds):
        #     print(f"pred: {pred}, truth: {truth}")
        # raise Exception("DEBUG")

        """ Metric evaluation. """
        # KL Divergence
        kl_loss = safe_kl_div_loss(
            pred_asv_log_abunds.unsqueeze(0),
            torch.log(true_asv_abunds.unsqueeze(0)),
        ).item()

        # Root-Mean-Square-Log error
        log_10_conversion = np.log10(np.e)
        rmsle_loss = torch.sqrt(torch.mean(torch.square(
            log_10_conversion * (pred_asv_log_abunds - torch.log(true_asv_abunds))
        ))).item()

        # Spearman rank correlation
        spearman = scipy.stats.spearmanr(
            torch.log(true_asv_abunds).numpy(),
            pred_asv_log_abunds.numpy()
        )

        # Bray-curtis
        bc = bray_curtis(true_asv_abunds.numpy(), torch.exp(pred_asv_log_abunds).numpy())

        df_entries.append({
            'SampleIdx': sample_idx,
            'KL_err': kl_loss,
            'RMSL_err': rmsle_loss,
            'SpearmanCorr': spearman.statistic,
            'BrayCurtis': bc,
            'NumSGB': len(true_asv_abunds),
        })
    return pd.DataFrame(df_entries)

## Baseline models

In [ ]:
def evaluate_baseline_model(
    baseline_method: BaselinePredictor,
    dset_object: ASVPreembeddedDataset,
) -> pd.DataFrame:
    """
    A wrapper for evaluate_method for baseline methods.

    This creates a function that implements the inference_fn for evaluate_method, and invokes it to return a table.
    :param baseline_method:
    :param dset_object:
    :return: dataframe of results.
    """
    def predict_fn(_dset: ASVPreembeddedDataset, _idx: int):
        asv_ids, _  = _dset.true_abundance_profile(_idx)
        # subset_df = dset_df.loc[dset_df.index == sample_id]
        # assert subset_df.shape[0] == 1, f"Expected 1 sample matching ID {sample_id}, got: {subset_df.shape[0]}"
        # extractor = MetaphlanProfileParser(subset_df)
        # target_sample = next(iter(extractor.samples()))

        # Note: the below code ensures that we only include SGBs that are present in the PhyloPhlAn marker DB.
        abunds = baseline_method.predict_abundances(asv_ids)
        abunds = torch.from_numpy(abunds + 1e-8).to(torch.float32)  # add padding so that spurious "zeroes" don't result in NaNs or infs.
        abunds = abunds / abunds.sum()   # renormalize.
        return torch.log(abunds)

    return evaluate_method(predict_fn, dset_object)

## torch models

In [ ]:
def perform_inference_torch_model(
        torch_model: nn.Module,
        dset: ASVPreembeddedDataset,
        sample_idx: int,
        eval_device: str = 'cuda'
) -> Tensor:
    """
    :param torch_model: A pytorch model which takes as input a batched tensor of SGB embeddings, which was trained using the function 'main_training_loop'. This model should output logits.
    """
    _, features, _ = dset[sample_idx]
    n_taxa = len(features)
    features = features.unsqueeze(0).to(eval_device)
    marker_mask = torch.ones(size=(1, n_taxa, 1), dtype=torch.bool, device=eval_device)
    taxa_mask = torch.ones(size=(1, n_taxa), dtype=torch.bool, device=eval_device)

    torch_model.eval()
    with torch.no_grad():
        nn_output = torch_model(features, marker_mask, taxa_mask).to("cpu").to(torch.float32)
        nn_output = nn_output[0]
        n_finite_nn = (~torch.isinf(nn_output)).sum().item()
        n_finite_expected = n_taxa
        assert n_finite_expected == n_finite_nn, f"Expected {n_finite_expected} entries of model output to all be finite, got: {n_finite_nn}"

        gc.collect()
        torch.cuda.empty_cache()
        log_prob_output = nn.functional.log_softmax(nn_output, dim=-1)    # log probabilities
    return log_prob_output


def evaluate_torch_model(
    model_savedir: Path,
    dset: ASVPreembeddedDataset,
    device: str = 'cuda',
) -> pd.DataFrame:
    """
    A wrapper around the previously-defined 'evaluate_method' function.

    This function was written so that we can load the model just once from a previously-saved pytorch model state file.
    """
    model_config_file = model_savedir / "model_config.json"
    metadata_file = model_savedir / "metadata.txt"
    assert model_config_file.exists(), f"Pytorch model config '{model_config_file}' does not exist!"
    assert metadata_file.exists(), f"Pytorch model training metadata '{metadata_file}' does not exist!"

    with open(metadata_file, "rt") as f:
        metadata = dict()
        for line_idx, line in enumerate(f):
            line = line.strip()
            tokens = line.split("=")
            assert len(tokens) == 2, f"Unparseable metadata, line {line_idx+1}: {line}"
            k, v = tokens
            metadata[k] = v
    assert "epochs" in metadata, f"Entry 'epochs' was not found in metadata ({metadata_file})"
    n_epochs = int(metadata["epochs"])

    checkpoint_files = {}
    for pt_file in (model_savedir / 'model_checkpoints').glob("checkpoint_*.pt"):
        checkpoint_number = int(pt_file.stem.split("_")[-1])
        checkpoint_files[checkpoint_number] = pt_file
    if len(checkpoint_files) == 0:
        raise ValueError(f"No saved checkpoint model states in {model_savedir}")

    # try to load the latest state.
    checkpoint_epochs = sorted(checkpoint_files.keys())
    last_checkpoint_epoch = checkpoint_epochs[-1]
    checkpoint_fpath = checkpoint_files[last_checkpoint_epoch]
    if last_checkpoint_epoch < n_epochs:
        print(f"Last trained epoch was {checkpoint_epochs[-1]}; loading last-known state file {checkpoint_fpath}")
    elif last_checkpoint_epoch > n_epochs:
        raise ValueError(f"Last trained epoch checkpoint {last_checkpoint_epoch} exceeds metadata epochs {n_epochs}. Dir: {model_savedir}")
    else:
        pass

    with open(model_config_file, "rt") as json_cfg:
        model_cfg = json.load(json_cfg)
        model_class_name = model_cfg['class']
        del model_cfg['class']  # unnecessary for evaluation.
        del model_cfg['init_rng_seed']  # unnecessary for evaluation.

    if model_class_name == "SGBAbundanceLayeredPredictionModel":
        model = SGBAbundanceLayeredPredictionModel(**model_cfg).to(device)
    elif model_class_name == "SGBAbundancePredictionModel":
        model = SGBAbundancePredictionModel(**model_cfg).to(device)
    elif model_class_name == "SGBEmbedPoolConcatPredictionModel":
        model = SGBEmbedPoolConcatPredictionModel(**model_cfg).to(device)
    else:
        raise KeyError("Unsupported model class name: {}".format(model_class_name))

    model = torch.compile(model)
    checkpoint: Dict[str, Any] = torch.load(checkpoint_fpath)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # plot loss history.
    epoch_history: List[int] = checkpoint['epoch_history']
    training_history: List[float] = checkpoint['training_history']
    test_history: List[float] = checkpoint['test_history']
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.plot(epoch_history, training_history, color='red', label='Training Loss')
    ax.plot(epoch_history, test_history, color='green', label='Test Loss')
    ax.set_ylabel("Loss")
    ax.set_xlabel("Epoch")
    ax.legend()
    plt.show()
    # plt.savefig(Path("results") / f"{checkpoint_fpath.stem}_loss.png", bbox_inches='tight')
    plt.close(fig)

    return evaluate_method(
        lambda _dset, _idx: perform_inference_torch_model(model, _dset, _idx, eval_device=device),
        dset
    )

# Evaluation & Benchmark

In [ ]:
embed_models = ['dnabert-s', 'evo-1-8k-base_hyena5', 'evo2_7b_hyena10']
pred_models = ['epc_nopool_kl', 'epc_pool_kl']
baseline_methods = {
    'uniform': UniformAbundancePredictor()
}

def evaluate_all(dset_name: str):
    abundance_table_dir = Path("/data/cctm/youn/human_microbiome_compendium/asv")
    dset_dir = Path("/data/bwh-comppath-seq/youn/human_microbiome_compendium") / dset_name
    embeddings_dir = Path("/data/bwh-comppath-seq/youn/human_microbiome_compendium") / dset_name / "embeddings"

    assert dset_dir.exists(), f"Dataset {dset_name} files not found! ({dset_dir})"
    assert embeddings_dir.exists(), f"Embeddings root dir for {dset_name} not found! ({embeddings_dir})"

    # train_df = pd.read_csv(dset_dir / "train.tsv", sep="\t")
    test_df = pd.read_csv(dset_dir / "test.tsv", sep="\t")

    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True, parents=True)

    test_dset_for_baseline = None
    for embed_model in embed_models:
        print(f"Embed model = {embed_model}")
        # train_dset = ASVPreembeddedDataset(train_df, abundance_table_dir, embedding_h5_path)
        embedding_h5_path = embeddings_dir / f'{embed_model}.h5'
        test_dset = ASVPreembeddedDataset(test_df, abundance_table_dir, embedding_h5_path)
        if test_dset_for_baseline is None:
            test_dset_for_baseline = test_dset

        for pred_model in pred_models:
            print(f"\tPred model = {pred_model}")
            model_savedir = dset_dir / "trained_models" / embed_model / pred_model
            evaluate_torch_model(
                model_savedir=model_savedir,
                dset=test_dset,
                device='cuda'
            ).to_csv(results_dir / f"{embed_model}__{pred_model}.tsv", sep='\t', index=False)

    # baseline methods
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for baseline_name, baseline_method in baseline_methods.items():
            evaluate_baseline_model(
                baseline_method=baseline_method,
                dset_object=test_dset_for_baseline,
            ).to_csv(results_dir / f'{baseline_name}.tsv', sep='\t', index=False)